# Step 10 — Score the Rule-Based Classifier

**Inputs**
- `data/validation/09_ground_truth.parquet` — the correct answers, from notebook 09
- `config.VALIDATION_BUILDINGS_FILE` — the buildings the reviewed sample was drawn from

**Outputs**
- `data/validation/10_score_detail.csv` — one row per building, with what was right and wrong
- `data/validation/10_score_summary.csv` — the headline numbers

## What this notebook does

1. Runs `rule_utils.classify_building` over the buildings a person reviewed.
2. Compares each prediction with the correct answer from notebook 09.
3. Reports how many labels were right, how many were extra, and how many were missed.

The classifier has never seen these answers, and none of its rules were written by
looking at them, so this is a fair test of how it behaves on unseen buildings.

## The two things being measured

**Activities** is a set, so a prediction can be partly right. We report **precision and
recall separately**, because the two kinds of mistake have opposite consequences:

| Mistake | What it does to the capacity model |
|---|---|
| predicting an activity that is not there | the building takes a share of the zone total it has no claim on |
| missing an activity that is there | the building drops out of that activity altogether |

Combining them into one F1 number would hide which is happening, and the difference
matters when the output feeds a transport model.

**Bosserhof class** is a single value, so it is either right or wrong — plain accuracy.

## Which buildings are used, and why it matters

Notebook 05 numbers buildings by their **row position**:

```python
gdf['gml_id'] = gdf.index        # notebook 05, cell 9
```

Those numbers only mean something within a single run. Running notebooks 01–05 again on
the same code and same inputs produced **567,961** buildings where the reviewed sample
came from a run of **578,080** — and of the ids that still existed in both, **none**
described the same building.

So this notebook uses the building file the sample was actually drawn from, and Step 3
confirms the ids still line up before anything is scored.

In [1]:
import sys
sys.path.insert(0, str(__import__('pathlib').Path('..').resolve()))
from config import (
    VALIDATION_GROUND_TRUTH, VALIDATION_SCORE_DETAIL, VALIDATION_SCORE_SUMMARY,
    VALIDATION_BUILDINGS_FILE, BUILDING_FUNCTION_CODELIST, ZONE_ACTIVITY_COLUMNS,
)
from rule_utils import classify_building, reachable_bosserhof_classes
from validation_utils import (
    collapse_to_zone_activities, score_activities, score_bosserhof,
    per_activity_breakdown, format_activity_report, format_bosserhof_report,
)

import pandas as pd
import pyogrio

pd.set_option('display.width', 170)
pd.set_option('display.max_colwidth', 55)


def as_set(value):
    """Coerce a ground-truth label cell to a set.

    Parquet round-trips list columns as numpy arrays, and `value or []` raises
    "truth value of an array is ambiguous" on those, so the None check has to be
    explicit rather than relying on falsiness.
    """
    return set() if value is None else set(value)


for path, what in [(VALIDATION_GROUND_TRUTH, 'notebook 09 output'),
                   (VALIDATION_BUILDINGS_FILE, 'the building dataset for the sample')]:
    if not path.exists():
        raise FileNotFoundError(f'{path}\n  missing: {what}')
print('Inputs present.')

Inputs present.


---
## Step 1 — Load the correct answers

Read the table notebook 09 produced and print how many buildings have a usable answer
for each dimension, together with how each answer was established. Buildings the reviewer
was unsure about are set aside here, and the printed counts show exactly how many and
why — so the totals in the results below can always be traced back.

In [2]:
truth = pd.read_parquet(VALIDATION_GROUND_TRUTH)
truth['gml_id'] = truth['gml_id'].astype(str)

_leaked = [c for c in truth.columns
           if any(t in c.lower() for t in ('predicted', 'prefill', 'replacement',
                                           'precision', 'recall', 'accuracy'))]
assert not _leaked, f'ground truth contains prediction or metric columns: {_leaked}'

print(f'{len(truth):,} annotated buildings, {len(truth.columns)} columns')
print(f'  activities scoreable : {int(truth["activities_scoreable"].sum()):,}')
print(f'  bosserhof  scoreable : {int(truth["bosserhof_scoreable"].sum()):,}')
print()
print('How each activity label was established:')
print(truth['activities_scoreable_reason'].value_counts().to_string())
print()
print('How each Bosserhof label was established:')
print(truth['bosserhof_scoreable_reason'].value_counts().to_string())

1,391 annotated buildings, 13 columns
  activities scoreable : 882
  bosserhof  scoreable : 874

How each activity label was established:
activities_scoreable_reason
human confirmed the label                     553
excluded: uncertain                           337
human typed a replacement                     316
excluded: unvalidated                         159
excluded: marked wrong but never corrected     13
human wrote: no activity here                  13

How each Bosserhof label was established:
bosserhof_scoreable_reason
human confirmed the class                                 719
excluded: uncertain                                       337
excluded: unvalidated                                     159
human typed a replacement                                 131
human wrote: no class                                      14
excluded: marked wrong but never corrected                 13
excluded: human named several classes, no single truth      8
single class extracted from fr

---
## Step 2 — Run the classifier

Load the buildings and classify each one with `rule_utils.classify_building`.

One detail to be aware of: the rule tables look up buildings by their **ALKIS code**
(for example `31001_2000`), but this building file was created before that column was
kept and only has the English label. So the code is recovered from the label first.

That is safe here because the two map one-to-one: 71 labels, 71 codes, none shared. The
cell checks this before relying on it, and stops with a clear message if a label ever
turns out to match more than one code.

The printout at the end shows which layer of the classifier answered for each building —
a POI tag, an ALKIS code, an OSM fallback, or nothing at all.

In [3]:
NEEDED = ['gml_id', 'volume_m3', 'label_en', 'amenity', 'shop', 'tourism', 'building',
          'information', 'additional_information', 'osm_building_type',
          'osm_landuse_class', 'osm_names']

fields = set(pyogrio.read_info(VALIDATION_BUILDINGS_FILE)['fields'])
cols = [c for c in NEEDED if c in fields]
src = pyogrio.read_dataframe(VALIDATION_BUILDINGS_FILE, columns=cols, read_geometry=False)
src['gml_id'] = src['gml_id'].astype(str)
print(f'{VALIDATION_BUILDINGS_FILE.name}: {len(src):,} buildings, {len(cols)} columns')

# --- recover the ALKIS code from the English label, one-to-one asserted ---
codelist = pd.read_csv(BUILDING_FUNCTION_CODELIST, encoding='utf-8',
                       encoding_errors='replace')
per_label = codelist.groupby('label_en')['function'].nunique()
ambiguous = set(per_label[per_label > 1].index)

present = set(src['label_en'].dropna().unique())
unknown = present - set(codelist['label_en'])
collides = present & ambiguous

if unknown:
    raise AssertionError(f'{len(unknown)} label_en values absent from the codelist: '
                         f'{sorted(unknown)[:5]}')
if collides:
    raise AssertionError(
        'label_en to function is NOT one-to-one here; these labels are shared by more '
        f'than one ALKIS code, so keying on them would mis-classify: {sorted(collides)}. '
        'Regenerate the input with `function` carried through instead of inferring it.')

print(f'  label_en to function: one-to-one verified over {len(present)} labels')
src['function'] = src['label_en'].map(dict(zip(codelist['label_en'],
                                              codelist['function'])))

# --- classify ---
pred = pd.DataFrame([classify_building(r) for r in src.to_dict('records')])
pred['gml_id'] = pred['gml_id'].astype(str)
pred['zone_activities'] = pred['mid_labels'].map(collapse_to_zone_activities)
pred = pred.merge(src[['gml_id', 'volume_m3', 'label_en', 'function']],
                  on='gml_id', how='left', validate='one_to_one')

n_cls = int(pred['bosserhof_class'].notna().sum())
print(f'\nclassified: {n_cls:,} / {len(pred):,} ({n_cls / len(pred):.1%})')
print('\nResolved by layer:')
for k, v in pred['interpreted_type'].value_counts().items():
    print(f'  {k:14s} {v:>8,} ({v / len(pred):5.1%})')

condensed_buildings_with_pois.gpkg: 578,080 buildings, 12 columns
  label_en to function: one-to-one verified over 71 labels

classified: 284,971 / 578,080 (49.3%)

Resolved by layer:
  alkis_code      521,384 (90.2%)
  no_signal        43,421 ( 7.5%)
  poi_tags         12,686 ( 2.2%)
  osm_fallback        589 ( 0.1%)


---
## Step 3 — Check we are scoring the right buildings

Before comparing anything, confirm that a building id in the answers table really refers
to the same building in the classified data.

Two checks:

1. **Do the ids exist?** How many reviewed buildings appear in the classified data at all.
2. **Are they the same buildings?** The spreadsheet recorded each building's `volume_m3`
   at review time. If the ids still point at the same buildings, those volumes must still
   match.

The second check is the important one. Ids are row positions, so a lookup can succeed
while pointing at a completely different building — and then every number below would be
confidently wrong. This has happened: a previous version matched **94.7 %** of ids while
**0 %** of them were the same building. Precision even looked slightly better under the
fault, so the results gave no hint anything was wrong.

If fewer than 95 % of volumes match, this cell stops the notebook rather than reporting
figures.

In [4]:
m = truth.merge(
    pred[['gml_id', 'mid_labels', 'zone_activities', 'bosserhof_class',
          'interpreted_type', 'volume_m3', 'label_en', 'function']]
        .rename(columns={'volume_m3': 'volume_m3_rule'}),
    on='gml_id', how='left', validate='one_to_one')

n_matched = int(m['interpreted_type'].notna().sum())
print(f'GUARD 1  annotated ids present in the classified set : '
      f'{n_matched:,} / {len(m):,} ({n_matched / len(m):.1%})')

both = m[m['volume_m3'].notna() & m['volume_m3_rule'].notna()]
rel_diff = ((both['volume_m3'] - both['volume_m3_rule']).abs()
            / both['volume_m3'].abs().clip(lower=1e-9))
agree = rel_diff < 1e-3
share = float(agree.mean()) if len(both) else 0.0
print(f'GUARD 2  volume matches the annotated value          : '
      f'{int(agree.sum()):,} / {len(both):,} ({share:.1%})')

if len(both) and share < 0.95:
    print('\nLargest disagreements:')
    print(both.assign(rel_diff=rel_diff).nlargest(5, 'rel_diff')[
        ['gml_id', 'volume_m3', 'volume_m3_rule', 'osm_names']].to_string(index=False))
    raise AssertionError(
        f'Only {share:.1%} of gml_ids have a matching volume. `gml_id` is a per-run '
        'positional index, so the annotated sample came from a different run than this '
        'input: the join succeeds but points at the wrong buildings. Refusing to score.')

if n_matched < len(m):
    print(f'\nNote: {len(m) - n_matched} annotated building(s) absent from the input; '
          'they are excluded from every metric below and counted in the exclusions.')
print('\nIDENTITY CONFIRMED - the scored buildings are the annotated ones.')

GUARD 1  annotated ids present in the classified set : 1,391 / 1,391 (100.0%)
GUARD 2  volume matches the annotated value          : 1,391 / 1,391 (100.0%)

IDENTITY CONFIRMED - the scored buildings are the annotated ones.


---
## Step 4 — Activities: right, extra, missing

Counting happens in **individual labels**, not buildings. A building hosting three
activities contributes three labels, so a partly-correct answer scores partly.

For each building we compare the two sets and sort every label into one of three boxes:

| | |
|---|---|
| **correct** | the classifier predicted it and the person recorded it |
| **extra** | the classifier predicted it, the person did not record it |
| **missing** | the person recorded it, the classifier did not predict it |

Add those up across all buildings and the two headline figures follow:

```
precision = correct / (correct + extra)     of the labels predicted, how many were right
recall    = correct / (correct + missing)   of the labels that exist, how many were found
```

**A worked example.** The person recorded `{Workers, Retail_Non-Daily}` and the classifier
predicted `{Workers, Retail_Daily}`:

- `Workers` → **correct**
- `Retail_Daily` → **extra** (predicted, not really there)
- `Retail_Non-Daily` → **missing** (really there, not predicted)

So this one building adds 1 correct, 1 extra and 1 missing to the totals.

In [5]:
act = m[m['activities_scoreable'] & m['interpreted_type'].notna()].copy()

print('Buildings excluded from activity scoring, and why:')
excl = m[~(m['activities_scoreable'] & m['interpreted_type'].notna())]
print(excl['activities_scoreable_reason'].where(
    excl['interpreted_type'].notna(),
    'excluded: not present in the classified input').value_counts().to_string())
print(f'\nScoring {len(act):,} buildings.\n')

act_pairs = [(r['gml_id'], set(r['zone_activities']), as_set(r['activities_truth']))
             for _, r in act.iterrows()]
act_metrics, act_rows = score_activities(act_pairs)
print(format_activity_report(act_metrics, 'RULE ENGINE - ACTIVITIES'))

Buildings excluded from activity scoring, and why:
activities_scoreable_reason
excluded: uncertain                           337
excluded: unvalidated                         159
excluded: marked wrong but never corrected     13

Scoring 882 buildings.

RULE ENGINE - ACTIVITIES
  Counting individual activity LABELS, not buildings.

  buildings scored                     : 882
  labels the human recorded  (truth)   : 1,526
  labels the engine predicted          : 1,418

    correct   (predicted AND true)     : 1,198
    extra     (predicted, NOT true)    : 220
    missing   (true, NOT predicted)    : 328

  precision = 1,198 / 1,418 = 84.5%     of the labels predicted, this share was right
  recall    = 1,198 / 1,526 = 78.5%     of the labels that exist, this share was found

  arithmetic check: 1,198 + 220 = 1,418 predicted   |  1,198 + 328 = 1,526 truth

  buildings matching the human exactly : 479 of 882 (54.3%)
    extra labels only                  : 116   (misallocates capacity)
 

### Which activities get invented, and which get overlooked

The headline numbers say how much was wrong; this table says **which activities** were
wrong and in which direction.

| Column | Meaning |
|---|---|
| `in_truth` | how many buildings really have this activity |
| `over_predicted` | how often it was claimed where it is not present |
| `missed` | how often it is present but was not claimed |
| `miss_rate` | `missed / in_truth` — the share of real occurrences that were overlooked |

`in_truth` is what makes the other columns readable: 20 misses out of 25 occurrences is a
very different situation from 20 out of 800.

In [6]:
breakdown = pd.DataFrame(per_activity_breakdown(act_pairs))
breakdown['miss_rate'] = (breakdown['missed']
                          / breakdown['in_truth'].where(breakdown['in_truth'] > 0)).round(3)
print(breakdown.sort_values('in_truth', ascending=False).to_string(index=False))

        activity  in_truth  over_predicted  missed  net_over  miss_rate
         Workers       850              21      58       -37      0.068
         Leisure       324              12     118      -106      0.364
Retail_Non-Daily       189               6     104       -98      0.550
    Retail_Daily        69             174      18       156      0.261
          School        50               4      15       -11      0.300
      University        25               2      10        -8      0.400
    Kindergarten        19               1       5        -4      0.263


---
## Step 5 — Bosserhof class

One value per building, so this is plain accuracy: the share of buildings where the
classifier named the same class the person did.

The cell also lists the buildings whose correct class **no rule can output**, and shows
what accuracy looks like with and without them. Those are two different problems — a
missing rule versus a rule choosing badly — and it helps to see them apart.

The confusion table at the end shows which classes get mixed up with which. It is often
more useful than the accuracy figure, because it points at specific rules to look at.

In [7]:
boss = m[m['bosserhof_scoreable'] & m['interpreted_type'].notna()].copy()
boss_pairs = [(r['gml_id'], r['bosserhof_class'], r['bosserhof_truth'])
              for _, r in boss.iterrows()]
boss_metrics, boss_rows = score_bosserhof(boss_pairs)
print(format_bosserhof_report(boss_metrics, 'RULE ENGINE - BOSSERHOF'))

blocked = boss[boss['bosserhof_truth_producible'] == False]
print(f'\nOf these, {len(blocked)} have a true class NO rule can produce '
      '(guaranteed miss - a vocabulary limit, not a rule error):')
for _, r in blocked.iterrows():
    print(f'   gml_id={r["gml_id"]:>8}  truth={r["bosserhof_truth"]!r:34} '
          f'rule said {r["bosserhof_class"]!r}')

REACHABLE = reachable_bosserhof_classes()
print(f'\nAccuracy excluding those {len(blocked)}: ', end='')
keep = boss[boss['bosserhof_truth_producible'] != False]
kp, _ = score_bosserhof([(r['gml_id'], r['bosserhof_class'], r['bosserhof_truth'])
                         for _, r in keep.iterrows()])
print(f'{kp["n_correct"]:,} / {kp["n_rows"]:,} = {kp["accuracy"]:.1%}')

wrong = pd.DataFrame(boss_rows)
wrong = wrong[wrong['match'] == 0]
if len(wrong):
    print('\nMost common confusions (rule predicted -> human said):')
    print(wrong.groupby(['predicted', 'truth']).size()
               .sort_values(ascending=False).head(12).to_string())

RULE ENGINE - BOSSERHOF — single label, plain accuracy
  rows scored            : 874
  correct                : 440
  accuracy               : 50.3%

Of these, 4 have a true class NO rule can produce (guaranteed miss - a vocabulary limit, not a rule error):
   gml_id=  550941  truth='garbage collection'               rule said 'public facilities'
   gml_id=  231906  truth='factory outlet centers'           rule said 'retail small scale'
   gml_id=  345680  truth='better school'                    rule said 'fitness wellness'
   gml_id=  556702  truth='fraternity'                       rule said 'public facilities'

Accuracy excluding those 4: 440 / 870 = 50.6%

Most common confusions (rule predicted -> human said):
predicted                         truth                                                           
services                          retail small scale                                                  24
industrial operations production  highly productive industries machine

---
## Step 6 — Results grouped by how much was known about the building

This is usually the most informative table in the notebook.

The classifier can only work with what the data gives it. A building tagged in OSM as a
bakery is easy; a building whose only description is a generic cadastral code meaning
"commercial premises" is impossible to place precisely, no matter how well the rules are
written.

Grouping the results by what was available separates those two situations, so a low
overall figure can be read as *what the data supports* rather than *how good the rules
are*:

| Group | What the classifier had to work with |
|---|---|
| 1 POI tag | an OSM tag naming the business type |
| 2 generic commercial ALKIS code | a cadastral code meaning only "commercial" |
| 3 specific ALKIS code | a cadastral code naming the building type |
| 4 OSM footprint / land use | only the footprint type or surrounding land use |
| 5 no usable signal | nothing |
| 6 ALKIS code says no activity | the cadastre says this is a dwelling |

In [8]:
GENERIC_COMMERCIAL = {'31001_2000', '31001_2010', '31001_1120', '31001_2310',
                      '31001_2100', '31001_1130', '31001_2320'}

def tier(row):
    if row['interpreted_type'] == 'poi_tags':     return '1 POI tag on the building'
    if row['interpreted_type'] == 'osm_fallback': return '4 OSM footprint / land use'
    if row['interpreted_type'] == 'no_signal':    return '5 no usable signal'
    if str(row.get('function') or '') in GENERIC_COMMERCIAL:
        return '2 generic commercial ALKIS code'
    if row['bosserhof_class']:                    return '3 specific ALKIS code'
    return '6 ALKIS code says no activity'

act['tier'] = act.apply(tier, axis=1)

rows = []
for name, grp in act.groupby('tier'):
    pairs = [(r['gml_id'], set(r['zone_activities']), as_set(r['activities_truth']))
             for _, r in grp.iterrows()]
    mt, _ = score_activities(pairs)
    correct = mt['n_true_positive']
    rows.append({'information available': name, 'buildings': mt['n_rows'],
                 'correct': correct, 'extra': mt['n_over_predicted'],
                 'missing': mt['n_missed'],
                 'precision': round(mt['precision'], 3),
                 'recall': round(mt['recall'], 3),
                 'exact_set': round(mt['exact_match_rate'], 3)})
print(pd.DataFrame(rows).sort_values('information available').to_string(index=False))

          information available  buildings  correct  extra  missing  precision  recall  exact_set
      1 POI tag on the building        435      784    158       94      0.832   0.893      0.611
2 generic commercial ALKIS code        265      253     12      115      0.955   0.688      0.581
          3 specific ALKIS code        110      158     49       27      0.763   0.854      0.500
     4 OSM footprint / land use          3        3      1        1      0.750   0.750      0.667
             5 no usable signal         12        0      0       15      0.000   0.000      0.000
  6 ALKIS code says no activity         57        0      0       76      0.000   0.000      0.035


---
## Step 7 — Save

Two files:

- **`10_score_detail.csv`** — one row per building: the correct answer, what the
  classifier predicted, and the per-building counts. Any figure above can be traced back
  to the individual buildings behind it.
- **`10_score_summary.csv`** — the headline numbers, for quoting elsewhere.

In [9]:
def join_labels(value):
    """Explicit marker for an empty set. ';'.join(set()) is '', which a CSV reader
    restores as NaN - indistinguishable from missing data, when it actually means
    the classifier asserted no activity here."""
    labels = sorted(as_set(value))
    return ';'.join(labels) if labels else '(no activity)'


detail = act[['gml_id', 'tier', 'interpreted_type', 'activities_scoreable_reason',
              'label_en', 'osm_names', 'volume_m3',
              'bosserhof_class', 'bosserhof_truth']].copy()
detail = detail.rename(columns={'bosserhof_class': 'bosserhof_rule',
                                'bosserhof_truth': 'bosserhof_human'})
detail['activities_human'] = act['activities_truth'].map(join_labels)
detail['activities_rule'] = act['zone_activities'].map(join_labels)

per_row = pd.DataFrame(act_rows).rename(columns={
    'key': 'gml_id', 'n_true_positive': 'n_correct',
    'n_over_predicted': 'n_extra', 'n_missed': 'n_missing'})
detail = detail.merge(per_row[['gml_id', 'n_correct', 'n_extra', 'n_missing']],
                      on='gml_id', how='left')
detail.to_csv(VALIDATION_SCORE_DETAIL, index=False)
print(f'detail  -> {VALIDATION_SCORE_DETAIL}  ({len(detail):,} rows)')

correct = act_metrics['n_true_positive']
summary = pd.DataFrame([
    {'dimension': 'activities', 'metric': 'buildings_scored',   'value': act_metrics['n_rows']},
    {'dimension': 'activities', 'metric': 'labels_human',       'value': correct + act_metrics['n_missed']},
    {'dimension': 'activities', 'metric': 'labels_predicted',   'value': correct + act_metrics['n_over_predicted']},
    {'dimension': 'activities', 'metric': 'correct',            'value': correct},
    {'dimension': 'activities', 'metric': 'extra',              'value': act_metrics['n_over_predicted']},
    {'dimension': 'activities', 'metric': 'missing',            'value': act_metrics['n_missed']},
    {'dimension': 'activities', 'metric': 'precision',          'value': round(act_metrics['precision'], 4)},
    {'dimension': 'activities', 'metric': 'recall',             'value': round(act_metrics['recall'], 4)},
    {'dimension': 'activities', 'metric': 'exact_set_match',    'value': round(act_metrics['exact_match_rate'], 4)},
    {'dimension': 'bosserhof',  'metric': 'buildings_scored',   'value': boss_metrics['n_rows']},
    {'dimension': 'bosserhof',  'metric': 'correct',            'value': boss_metrics['n_correct']},
    {'dimension': 'bosserhof',  'metric': 'accuracy',           'value': round(boss_metrics['accuracy'], 4)},
    {'dimension': 'bosserhof',  'metric': 'truth_not_producible', 'value': int(len(blocked))},
])
summary.to_csv(VALIDATION_SCORE_SUMMARY, index=False)
print(f'summary -> {VALIDATION_SCORE_SUMMARY}')
print()
print(summary.to_string(index=False))

detail  -> C:\Users\Mayur Patel\Documents\GitHub\Capacity_Calculation-pipeline-optimized\data\validation\10_score_detail.csv  (882 rows)
summary -> C:\Users\Mayur Patel\Documents\GitHub\Capacity_Calculation-pipeline-optimized\data\validation\10_score_summary.csv

 dimension               metric     value
activities     buildings_scored  882.0000
activities         labels_human 1526.0000
activities     labels_predicted 1418.0000
activities              correct 1198.0000
activities                extra  220.0000
activities              missing  328.0000
activities            precision    0.8449
activities               recall    0.7851
activities      exact_set_match    0.5431
 bosserhof     buildings_scored  874.0000
 bosserhof              correct  440.0000
 bosserhof             accuracy    0.5034
 bosserhof truth_not_producible    4.0000


---
## How to read these numbers

**Read precision and recall separately.** They describe different problems. Extra labels
give a building a share of activity it should not have; missing labels drop it from that
activity altogether. Averaging the two into an F1 score hides which one is happening.

**The per-activity table usually says more than the headline.** For example a large
`over_predicted` count on `Retail_Daily` comes mostly from the label taxonomy: the
`errands` label collapses into the daily-shopping column, so errand trips show up as
shopping. That is a mapping choice, not a faulty rule.

**Group 6 in the Step 6 table scores zero by design.** Those buildings have a cadastral
code stating they are dwellings, so the classifier reports no activity. Where the reviewer
found a business inside, the cadastre is out of date — a data currency issue rather than a
classification mistake, and one that affects any method reading the same source.

**Bosserhof mistakes come in two kinds.** Some are the rules picking the wrong class,
which better rules would fix. Others are classes no rule can output at all, which need a
new rule. Step 5 separates them.